# Predicting subsurface eddy tilt from surface signatures

This workflow tests AE and CE independently. It reserves a final set of unseen eddies before any model, target formulation, hyperparameter, or feature-set selection. Training gives each eddy equal total weight so long-lived eddies do not dominate.

Two target formulations compete during grouped training-only cross-validation:

1. direct eastward and northward tilt components;
2. `log1p(TiltDis)` and the circular directional offset from the PV-gradient bearing.

The final test eddies are evaluated exactly once after all choices are fixed.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import seacofs_tilt_tools as tilt
import ml_subsurface_tools as ml

pd.set_option("display.max_columns", 50)
plt.rcParams["figure.dpi"] = 120

RANDOM_STATE = 42
TEST_SIZE = 0.20
INNER_FOLDS = 5
REPEATED_SPLITS = 5

## 1. Load data and enforce a common vertical interval

Tilt is capped at the approximately 859 m model level. Only eddy-days with a vertical profile deeper than 850 m are retained, so every target represents the complete measured interval. Filtering occurs before the relatively expensive core-mean PV calculation.

In [ ]:
paths = tilt.Paths()
grid = tilt.load_grid(paths.grid, paths.z_r)
df_eddies, df_tilt = tilt.load_tilt_tables(paths)
df_vert = tilt.load_vert(paths)

max_depth = (
    df_vert.groupby(["Eddy", "Day"], as_index=False)["Depth"]
    .max()
    .rename(columns={"Depth": "max_profile_depth"})
)
df_eddies = df_eddies.merge(max_depth, on=["Eddy", "Day"], how="inner")
df_eddies = df_eddies[df_eddies["max_profile_depth"] > 850].copy()
df_eddies = tilt.add_pv_gradient_terms(df_eddies, grid, core_mean=True)

print(f"Rows reaching the full depth: {len(df_eddies):,}")
print(f"Eddies reaching the full depth: {df_eddies['Eddy'].nunique():,}")

## 2. Engineer predictors and audit target coverage

The canonical predictors are:

```text
beta, h, Omega, Rc, AR, norm_time
prop_east_km_day, prop_north_km_day
ellipse_major_cos2, ellipse_major_sin2
Rc_tendency_km_day, Omega_tendency_day
PV_grad_mag, PV_grad_east, PV_grad_north
```

Only `beta` is used from the correlated latitude/Coriolis group; `lat` and `f` are excluded. `Omega` is used and `w` is excluded. Ellipse orientation uses `sin(2θ)` and `cos(2θ)` because an ellipse axis is 180° periodic. Dynamic predictors are daily differences within each eddy. Missing first-day dynamics are imputed inside each training fold.

In [ ]:
display(ml.target_availability_summary(df_eddies))

model_df = ml.prepare_modelling_table(df_eddies)
polarity_data = {cyc: model_df[model_df["Cyc"] == cyc].copy() for cyc in ("AE", "CE")}

display(pd.DataFrame({
    cyc: {"rows": len(part), "eddies": part["Eddy"].nunique()}
    for cyc, part in polarity_data.items()
}).T)
display(pd.Series(ml.FEATURES, name="model_feature").to_frame())

In [ ]:
for cyc, part in polarity_data.items():
    part[ml.FEATURES + ["TiltDis"]].hist(bins=50, figsize=(15, 12))
    plt.suptitle(f"{cyc}: engineered predictors and tilt distance", y=1.01)
    plt.tight_layout()

## 3. Reserve untouched final-test eddies

AE and CE are split independently. Every observation from an eddy stays on one side of the split. Nothing below the next model-selection section may use the final test observations until Part 7.

In [ ]:
splits = {
    cyc: ml.grouped_train_test_split(part, test_size=TEST_SIZE, random_state=RANDOM_STATE)
    for cyc, part in polarity_data.items()
}

summary = []
for cyc, split in splits.items():
    assert set(split.groups_train).isdisjoint(set(split.groups_test))
    summary.append({
        "Cyc": cyc, "training rows": len(split.train), "test rows": len(split.test),
        "training eddies": split.groups_train.nunique(),
        "untouched test eddies": split.groups_test.nunique(),
    })
pd.DataFrame(summary).set_index("Cyc")

## 4. Training-only grouped model and hyperparameter selection

Five-fold grouped CV compares Ridge regularisation, four restrained gradient-boosting configurations, and both target formulations. Each fold includes a mean-vector and PV-direction baseline. Selection uses equal-eddy-weighted vector MAE. This is the most computationally intensive section.

In [ ]:
candidates = ml.default_candidates()
tuning = {}
selected = {}

for cyc, split in splits.items():
    print(f"Tuning {cyc}...")
    tuning[cyc] = ml.tune_candidates_grouped(
        split.train, candidates=candidates, n_splits=INNER_FOLDS, random_state=RANDOM_STATE
    )
    selected[cyc] = ml.best_candidate(tuning[cyc], candidates)
    print(cyc, selected[cyc])
    display(ml.summarise_tuning(tuning[cyc]).head(12).round(3))

## 5. Training-only feature ablation

Using each polarity's selected model configuration, grouped CV tests whether PV magnitude, PV components, all PV terms, the added dynamic features, and `AR` improve generalisation. The final test set remains untouched.

In [ ]:
ablation = {}
chosen_features = {}
for cyc, split in splits.items():
    ablation[cyc] = ml.feature_ablation_cv(
        split.train, selected[cyc], n_splits=INNER_FOLDS, random_state=RANDOM_STATE
    )
    ablation_summary = (
        ablation[cyc].groupby("feature_set")
        .agg(eddy_vector_MAE=("eddy_weighted_vector_MAE_km", "mean"),
             fold_SD=("eddy_weighted_vector_MAE_km", "std"),
             distance_MAE=("distance_MAE_km", "mean"),
             angular_error=("median_angular_error_deg", "median"))
        .sort_values("eddy_vector_MAE")
    )
    best_set = ablation_summary.index[0]
    chosen_features[cyc] = ml.FEATURE_SETS[best_set]
    print(f"{cyc} selected feature set: {best_set}")
    display(ablation_summary.round(3))
    ablation_summary["eddy_vector_MAE"].sort_values().plot.barh(title=f"{cyc}: feature ablation")
    plt.xlabel("Training-CV equal-eddy vector MAE (km)")
    plt.show()

## 6. Repeated grouped validation inside the training population

Five additional grouped holdouts test sensitivity to which training eddies are selected. Baselines are recomputed inside every repeat. These repeats use only the original training population.

In [ ]:
repeated = {}
for cyc, split in splits.items():
    repeated[cyc] = ml.repeated_grouped_validation(
        split.train, selected[cyc], features=chosen_features[cyc],
        n_repeats=REPEATED_SPLITS, random_state=100,
    )
    display(repeated[cyc].groupby("model").agg(
        vector_MAE=("eddy_weighted_vector_MAE_km", "mean"),
        split_SD=("eddy_weighted_vector_MAE_km", "std"),
        distance_MAE=("eddy_weighted_distance_MAE_km", "mean"),
        angular_error=("median_angular_error_deg", "median"),
    ).sort_values("vector_MAE").round(3))

## 7. One final evaluation on untouched eddies

All model and feature choices are now fixed. Each selected model is fitted to all training eddies and evaluated once against its untouched final-test eddies. Both row-weighted and equal-eddy-weighted errors are reported.

In [ ]:
final_results = {}
for cyc, split in splits.items():
    fitted, prediction, scores, predictions = ml.fit_selected_and_test(
        selected[cyc], split, features=chosen_features[cyc], random_state=RANDOM_STATE
    )
    final_results[cyc] = {
        "fitted": fitted, "prediction": prediction,
        "scores": scores, "predictions": predictions,
    }
    print(f"{cyc}: {selected[cyc]}")
    print(f"Features: {chosen_features[cyc]}")
    display(scores.round(3))
    fig, axes = ml.plot_model_comparison(scores)
    fig.suptitle(f"{cyc}: untouched final-test eddies", y=1.03)

In [ ]:
for cyc, result in final_results.items():
    ml.plot_prediction_diagnostics(result["prediction"], title=f"{cyc}: selected model")
    display(ml.direction_performance_by_tilt(
        result["prediction"], thresholds=(0, 5, 10, 20)
    ).round(3))

## 8. Final-test permutation importance

Importance measures the increase in equal-eddy-weighted vector MAE after shuffling one selected feature in the untouched test set. It measures predictive reliance, not causality.

In [ ]:
importance_results = {}
for cyc, result in final_results.items():
    importance = ml.raw_feature_permutation_importance(
        result["fitted"], splits[cyc].test, selected[cyc],
        features=chosen_features[cyc], n_repeats=10, random_state=RANDOM_STATE,
    )
    importance_results[cyc] = importance
    display(importance.round(4))
    ml.plot_permutation_importance(importance, title=f"{cyc}: final-test importance")

## 9. Interpretation checklist

A polarity-specific ML approach is supported only if the selected model improves consistently over both fold-specific baselines during tuning, repeated training-only validation, and the untouched final test. Prioritise equal-eddy-weighted vector MAE; then inspect distance and thresholded angular skill.

Report for AE and CE separately:

- the selected model family, hyperparameters, target formulation, and feature set;
- training-CV and repeated-split performance relative to both baselines;
- untouched test performance, including equal-eddy-weighted metrics;
- whether the PV-relative target improves direction;
- whether PV magnitude adds information beyond its components;
- whether the propagation, ellipse, `Rc` tendency, and `Omega` tendency features improve CV;
- performance for larger tilts, where direction is better defined.

Do not interpret permutation importance as causal evidence. A negative result remains scientifically useful: it indicates that the tested surface signatures do not constrain instantaneous 859 m tilt sufficiently for unseen eddies.